In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('student.csv')




In [ ]:
df.isnull().sum()

In [3]:
df.duplicated().sum()

0

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
print("Categories in gender variables: ", end=" ")
print(df['gender'].unique())
print("Categories in race variables: ", end=" ")
print(df['race/ethnicity'].unique())
print("Categories in lunch variables: ", end=" ")
print(df['lunch'].unique())
print("Categories in test preparation course variables: ", end=" ")
print(df['test preparation course'].unique())
print("Categories in parental level of education variables: ", end=" ")
print(df['parental level of education'].unique())


In [ ]:
numeric = [feature for feature in df.columns if df[feature].dtype != 'O']
category = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Numeric features are:', numeric)
print('Categorical features are: ', category)

In [ ]:
df['total'] = df['math score'] + df['reading score'] + df['writing score']
df['average'] = df['total'] / 3
df.head()

In [ ]:
math_full = df[df['math score'] == 100]['average'].count()
reading_full = df[df['reading score']== 100]['average'].count()
writing_full = df[df['writing score']== 100]['average'].count()
print(f'Number of students with full marks in math {math_full}')
print(f'Number of students with full marks in reading {reading_full}')
print(f'Number of students with full marks in writing {writing_full}')




In [ ]:
math_less_20 = df[df['math score'] < 20]['average'].count()
reading_less_20 = df[df['reading score'] < 20]['average'].count()
writing_less_20 = df[df['writing score'] < 20]['average'].count()
print(f'Number of students less than 20 marks in math {math_less_20}')
print(f'Number of students less than 20 marks in reading {reading_less_20}')
print(f'Number of students less than 20 marks in writing {writing_less_20}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 7))

# First subplot
sns.histplot(data=df, x='average', bins=30, kde=True, color='g', ax=axs[0])

# Second subplot
sns.histplot(data=df, x='average', kde=True, hue='gender', ax=axs[1])

plt.tight_layout()
plt.show()


In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (15, 8))
plt.subplot(121)
sns.histplot(data = df, x = 'average', bins = 30, kde = True, color='r')
plt.subplot(122)
sns.histplot(data = df, x = 'average', kde = True, hue= 'gender')


In [14]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRFRegressor
import warnings



In [ ]:
df.head()

In [ ]:
df = df.drop(['total', 'average'], axis = 1)
df.head()

In [ ]:
X = df.drop('math score', axis = 1)


In [17]:
y = df['math score']


In [ ]:
numeric_features = X.select_dtypes(exclude='object').columns
category_features = X.select_dtypes(include='object').columns
print('Numerical features are: ', numeric_features)
print('Categorical features are: ', category_features)


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
scaler = StandardScaler()
ohe = OneHotEncoder()
ctm = ColumnTransformer(
    [
      ("OneHotEncoder", ohe, category_features),
        ("StandardScaler", scaler, numeric_features)
    ])

X = ctm.fit_transform(X)


In [ ]:
X.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=101)
y_test.shape

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)
    return mae, mse, rmse, r2


In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "KNeighborsRegressor": KNeighborsRegressor(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "AdaBoostRegressor": AdaBoostRegressor(),
    "Xgboost": XGBRFRegressor()
}

model_list = []
r2_list = []
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
   

    print(list(models.keys())[i])
    model_list.append((list(models.keys())[i]))

    print('Result for training set:')
    print('Model train mae:', model_train_mae)
    print('Model train mse:', model_train_mse)
    print('Model train rmse:', model_train_rmse)
    print('Model train r2:', model_train_r2)
    print('-------------------')
    print('Result for test set:')
    print('Model test mae:', model_test_mae)
    print('Model test mse:', model_test_mse)
    print('Model test rmse:', model_test_rmse)
    print('Model test r2:', model_test_r2)
    r2_list.append(model_test_r2)
    




In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns= ['Model_name', 'R2_score']).sort_values(by = ['R2_score'], ascending= False)